In [11]:
import os
import sys
from pathlib import Path

# Setup Colab environment
if 'google.colab' in sys.modules:
    repo_name = 'oc-hosp-surge-bn'
    repo_path = Path(f'/content/{repo_name}')

    if not repo_path.exists():
        !git clone https://github.com/AnhJoe/{repo_name}.git
    
    os.chdir(repo_path)
    !git pull origin main
    !apt-get install -y graphviz > /dev/null
    !pip install -q -r requirements.txt

# Find project root via src folder
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

# Configure path and working directory
if (root / "src").exists():
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    os.chdir(root)
    print(f"Project root: {root}")
else:
    print("Warning: 'src' folder not found.")

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 892 bytes | 892.00 KiB/s, done.
From https://github.com/AnhJoe/oc-hosp-surge-bn
 * branch            main       -> FETCH_HEAD
   beb16ae..3b9ae82  main       -> origin/main
Updating beb16ae..3b9ae82
Fast-forward
 notebooks/01_eda.ipynb | 62 +++++++++++++++++++++-----------------------------
 requirements.txt       |  6 +++--
 2 files changed, 30 insertions(+), 38 deletions(-)
Project root: /content/oc-hosp-surge-bn


In [13]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import glob
import re

# Analysis and modeling
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Project-specific EDA utilities
from src.eda import (
    choose_k_gap_rule,
    evaluate_kmeans_k,
    fit_kmeans,
    pca_project,
    plot_clusters_pca,
    plot_k_diagnostics,
)

# Global visualization styling
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.title_fontsize": 9,
    "legend.frameon": True,
    "legend.borderpad": 0.3,
})

In [ ]:
import os
import gdown

# Define the absolute root path
target_root = os.path.abspath('/data/raw')
folder_id = '1hkw97vM9arXhGRFRMZXundV16FBBeD0H'

if not os.path.exists(target_root):
    os.makedirs(target_root, exist_ok=True)
    print(f"Created directory at: {target_root}")
    # Download into the absolute root path
    !gdown --folder {folder_id} -O {target_root}
else:
    print(f"Directory already exists at: {target_root}")

# Verify file presence in the root directory
print("Files in root data directory:", os.listdir(target_root))

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/oc-resilience-bayes/data/raw/ca-hcai-preventablehospitalizations-county.csv'

In [ ]:
# Updated loading logic for the master dataset
hcai_path = os.path.join(target_root, 'ca-hcai-preventablehospitalizations-county.csv')
df_preventable_raw = pd.read_csv(hcai_path)

In [ ]:
# Map PDD files from 2017-2024 using the exported CSV naming convention
# Note: 2017 filename includes the specific date suffix from the source
pdd_filenames = {
    2017: '2017pddpivot.xlsx - Data.csv',
    2018: '2018pddpivot.xlsx - Data.csv',
    2019: '2019pddpivot.xlsx - Data.csv',
    2020: '2020pddpivot.xlsx - Data.csv',
    2021: '2021pddpivot.xlsx - Data.csv',
    2022: '2022pddpivot.xlsx - Data.csv',
    2023: '2023pddpivot.xlsx - Data.csv',
    2024: '2024pddpivot.xlsx - Data.csv'
}

# Dictionary to store raw DataFrames for inspection
raw_pdd_collection = {}

for year, filename in pdd_filenames.items():
    file_path = os.path.join('data/raw', filename)
    if os.path.exists(file_path):
        raw_pdd_collection[year] = pd.read_csv(file_path)
        print(f"Successfully loaded {year} PDD data")
    else:
        print(f"File missing: {file_path}")

In [ ]:
# Inspect a sample year from the PDD collection
raw_pdd_collection[2024].info()